
# Vision Transformer for CIFAR-10/100 Classification

Architecture Enhancements and Justifications:
1. Stochastic Depth: Improves regularization and training efficiency
2. Convolutional stem: two 3x3 stride-2 convolutions turn the 32x32 image into an 8x8 grid of tokens
3. Cosine Annealing with Warmup: Better convergence and generalization
4. LayerScale: Stabilizes training for deeper models
5. Advanced Data Augmentation: AutoAugment, CutMix, MixUp for robustness
6. Label Smoothing: Prevents overconfidence and improves generalization

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
from torch.cuda.amp import autocast, GradScaler

# Mount Google Drive (manual execution required)
# from google.colab import drive
# drive.mount('/content/drive')

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")

# Configuration
class Config:
    def __init__(self):
        # Student Information

        # Model Architecture
        self.IMG_SIZE = 32
        self.PATCH_SIZE = 4
        self.NUM_PATCHES = (self.IMG_SIZE // self.PATCH_SIZE) ** 2
        self.EMBED_DIM = 512
        self.NUM_HEADS = 8
        self.NUM_LAYERS = 12
        self.MLP_RATIO = 4
        self.DROPOUT = 0.1
        self.ATTENTION_DROPOUT = 0.1
        self.DROP_PATH = 0.1

        # Training Hyperparameters
        self.BATCH_SIZE = 128
        self.EPOCHS = 200
        self.LEARNING_RATE = 1e-3
        self.WEIGHT_DECAY = 0.05
        self.WARMUP_EPOCHS = 10
        self.LABEL_SMOOTHING = 0.1

        # Data Augmentation
        self.USE_CUTMIX = True
        self.USE_MIXUP = True
        self.CUTMIX_PROB = 0.5
        self.MIXUP_ALPHA = 0.2

        # Optimization
        self.USE_MIXED_PRECISION = True
        self.GRAD_CLIP = 1.0

# Stochastic Depth (DropPath)
# Justification: Randomly drops entire residual paths during training,
# improving regularization and reducing overfitting
class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        output = x.div(keep_prob) * random_tensor
        return output

# Multi-Head Self-Attention with improvements
# Justification: Core mechanism for capturing global dependencies
class MultiHeadAttention(nn.Module):
    def __init__(self, dim, num_heads=8, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

# MLP with GELU activation
# Justification: GELU provides smoother gradients than ReLU
class MLP(nn.Module):
    def __init__(self, in_features, hidden_features, drop=0.0):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, in_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

# Transformer Block with LayerScale
# Justification: LayerScale improves training stability for deeper models
class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, drop=0.0, attn_drop=0.0, drop_path=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = MultiHeadAttention(dim, num_heads, attn_drop, drop)
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = MLP(dim, int(dim * mlp_ratio), drop)

        # LayerScale for improved training stability
        self.gamma_1 = nn.Parameter(torch.ones(dim) * 1e-4)
        self.gamma_2 = nn.Parameter(torch.ones(dim) * 1e-4)

    def forward(self, x):
        x = x + self.drop_path(self.gamma_1 * self.attn(self.norm1(x)))
        x = x + self.drop_path(self.gamma_2 * self.mlp(self.norm2(x)))
        return x

# Vision Transformer with Advanced Features
class VisionTransformer(nn.Module):
    def __init__(self, config, num_classes=10):
        super().__init__()
        self.config = config
        self.num_classes = num_classes

        # Patch Embedding with Convolutional Stem
        # Justification: Conv stem provides better low-level feature extraction
        self.patch_embed = nn.Sequential(
            nn.Conv2d(3, config.EMBED_DIM // 2, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(config.EMBED_DIM // 2),
            nn.GELU(),
            nn.Conv2d(config.EMBED_DIM // 2, config.EMBED_DIM, kernel_size=3, stride=2, padding=1),
        )

        # Positional Embedding
        # Justification: Learnable positional embeddings adapt to the data
        self.pos_embed = nn.Parameter(torch.zeros(1, config.NUM_PATCHES + 1, config.EMBED_DIM))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, config.EMBED_DIM))
        self.pos_drop = nn.Dropout(p=config.DROPOUT)

        # Stochastic Depth decay rule
        dpr = [x.item() for x in torch.linspace(0, config.DROP_PATH, config.NUM_LAYERS)]

        # Transformer Blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(
                config.EMBED_DIM,
                config.NUM_HEADS,
                config.MLP_RATIO,
                config.DROPOUT,
                config.ATTENTION_DROPOUT,
                dpr[i]
            ) for i in range(config.NUM_LAYERS)
        ])

        self.norm = nn.LayerNorm(config.EMBED_DIM, eps=1e-6)

        # Classification Head with Dropout
        self.head = nn.Sequential(
            nn.Linear(config.EMBED_DIM, config.EMBED_DIM),
            nn.GELU(),
            nn.Dropout(config.DROPOUT),
            nn.Linear(config.EMBED_DIM, num_classes)
        )

        self._init_weights()

    def _init_weights(self):
        # Justification: Proper initialization critical for stable training
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_module_weights)

    def _init_module_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x):
        B = x.shape[0]

        # Patch Embedding
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)

        # Add CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)

        # Add positional embedding
        x = x + self.pos_embed
        x = self.pos_drop(x)

        # Transformer blocks
        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        # Classification
        x = self.head(x[:, 0])
        return x

# Advanced Data Augmentation
# Justification: Strong augmentation crucial for small datasets like CIFAR
def get_transforms(is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
            transforms.RandomErasing(p=0.25),
        ])
    else:
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
        ])

# CutMix and MixUp
# Justification: Improves generalization by training on interpolated samples
def cutmix_data(x, y, alpha=1.0):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]

    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (x.size()[-1] * x.size()[-2]))
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

def mixup_data(x, y, alpha=1.0):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# Training Function with Mixed Precision
def train_epoch(model, loader, optimizer, criterion, device, config, epoch, scaler):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{config.EPOCHS} [Train]')
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)

        # Apply CutMix or MixUp
        r = np.random.rand(1)
        if config.USE_CUTMIX and r < config.CUTMIX_PROB:
            inputs, targets_a, targets_b, lam = cutmix_data(inputs, targets, alpha=1.0)

            with autocast(enabled=config.USE_MIXED_PRECISION):
                outputs = model(inputs)
                loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        elif config.USE_MIXUP and r < config.CUTMIX_PROB + 0.5 * (1 - config.CUTMIX_PROB):
            inputs, targets_a, targets_b, lam = mixup_data(inputs, targets, config.MIXUP_ALPHA)

            with autocast(enabled=config.USE_MIXED_PRECISION):
                outputs = model(inputs)
                loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        else:
            with autocast(enabled=config.USE_MIXED_PRECISION):
                outputs = model(inputs)
                loss = criterion(outputs, targets)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

        pbar.set_postfix({
            'loss': f'{running_loss/(pbar.n+1):.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })

    return running_loss / len(loader), 100.0 * correct / total

# Evaluation Function
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in tqdm(loader, desc='[Eval]'):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    return running_loss / len(loader), 100.0 * correct / total

# Cosine Annealing with Warmup
# Justification: Warmup prevents instability in early training, cosine annealing improves convergence
class CosineAnnealingWarmupRestarts:
    def __init__(self, optimizer, max_lr, min_lr, warmup_steps, total_steps):
        self.optimizer = optimizer
        self.max_lr = max_lr
        self.min_lr = min_lr
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.current_step = 0

    def step(self):
        self.current_step += 1
        if self.current_step < self.warmup_steps:
            lr = self.max_lr * self.current_step / self.warmup_steps
        else:
            progress = (self.current_step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
            lr = self.min_lr + (self.max_lr - self.min_lr) * 0.5 * (1 + np.cos(np.pi * progress))

        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr

        return lr

# Main Training Loop
def train_model(dataset_name='CIFAR10', config=None):
    if config is None:
        config = Config()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset_name}")
    print(f"Device: {device}")
    print(f"{'='*60}\n")

    # Load Dataset
    if dataset_name == 'CIFAR10':
        train_dataset = torchvision.datasets.CIFAR10(
            root='./data', train=True, download=True, transform=get_transforms(True)
        )
        test_dataset = torchvision.datasets.CIFAR10(
            root='./data', train=False, download=True, transform=get_transforms(False)
        )
        num_classes = 10
    else:
        train_dataset = torchvision.datasets.CIFAR100(
            root='./data', train=True, download=True, transform=get_transforms(True)
        )
        test_dataset = torchvision.datasets.CIFAR100(
            root='./data', train=False, download=True, transform=get_transforms(False)
        )
        num_classes = 100

    train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # Initialize Model
    model = VisionTransformer(config, num_classes).to(device)

    # Loss and Optimizer
    criterion = nn.CrossEntropyLoss(label_smoothing=config.LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

    # Learning Rate Scheduler
    total_steps = config.EPOCHS * len(train_loader)
    warmup_steps = config.WARMUP_EPOCHS * len(train_loader)
    scheduler = CosineAnnealingWarmupRestarts(optimizer, config.LEARNING_RATE, 1e-6, warmup_steps, total_steps)

    # Mixed Precision Scaler
    scaler = GradScaler(enabled=config.USE_MIXED_PRECISION)

    # Training History
    history = {
        'train_loss': [], 'train_acc': [],
        'test_loss': [], 'test_acc': [],
        'lr': []
    }

    best_acc = 0.0
    start_time = time.time()

    # Training Loop
    for epoch in range(config.EPOCHS):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device, config, epoch, scaler)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)

        # Update scheduler per batch
        for _ in range(len(train_loader)):
            current_lr = scheduler.step()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        history['lr'].append(current_lr)

        print(f"\nEpoch {epoch+1}/{config.EPOCHS}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")
        print(f"Learning Rate: {current_lr:.6f}")

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(model.state_dict(), f'best_vit_{dataset_name.lower()}.pth')
            print(f"Best model saved with accuracy: {best_acc:.2f}%")

    total_time = time.time() - start_time
    print(f"\nTraining completed in {total_time/60:.2f} minutes")
    print(f"Best Test Accuracy: {best_acc:.2f}%")

    return history, best_acc

# Plot Training History
def plot_history(history, dataset_name):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Loss Plot
    axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2)
    axes[0, 0].plot(history['test_loss'], label='Test Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch', fontsize=12)
    axes[0, 0].set_ylabel('Loss', fontsize=12)
    axes[0, 0].set_title(f'{dataset_name} - Loss Curves', fontsize=14, fontweight='bold')
    axes[0, 0].legend(fontsize=10)
    axes[0, 0].grid(True, alpha=0.3)

    # Accuracy Plot
    axes[0, 1].plot(history['train_acc'], label='Train Accuracy', linewidth=2)
    axes[0, 1].plot(history['test_acc'], label='Test Accuracy', linewidth=2)
    axes[0, 1].set_xlabel('Epoch', fontsize=12)
    axes[0, 1].set_ylabel('Accuracy (%)', fontsize=12)
    axes[0, 1].set_title(f'{dataset_name} - Accuracy Curves', fontsize=14, fontweight='bold')
    axes[0, 1].legend(fontsize=10)
    axes[0, 1].grid(True, alpha=0.3)

    # Learning Rate Plot
    axes[1, 0].plot(history['lr'], linewidth=2, color='orange')
    axes[1, 0].set_xlabel('Batch Step', fontsize=12)
    axes[1, 0].set_ylabel('Learning Rate', fontsize=12)
    axes[1, 0].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)

    # Summary Table
    axes[1, 1].axis('off')
    summary_text = f"""
    Results Summary

    Dataset: {dataset_name}

    Final Train Acc: {history['train_acc'][-1]:.2f}%
    Final Test Acc: {history['test_acc'][-1]:.2f}%
    Best Test Acc: {max(history['test_acc']):.2f}%

    Final Train Loss: {history['train_loss'][-1]:.4f}
    Final Test Loss: {history['test_loss'][-1]:.4f}
    """
    axes[1, 1].text(0.1, 0.5, summary_text, fontsize=11, verticalalignment='center', family='monospace')

    plt.tight_layout()
    plt.savefig(f'{dataset_name.lower()}_training_results.png', dpi=300, bbox_inches='tight')
    plt.show()

# Hyperparameter Comparison
def compare_hyperparameters():
    """
    Compare different hyperparameter configurations
    """
    configs = {
        'Baseline': {
            'EMBED_DIM': 384, 'NUM_LAYERS': 8, 'NUM_HEADS': 6,
            'LEARNING_RATE': 1e-3, 'BATCH_SIZE': 128
        },
        'Deep Model': {
            'EMBED_DIM': 512, 'NUM_LAYERS': 12, 'NUM_HEADS': 8,
            'LEARNING_RATE': 1e-3, 'BATCH_SIZE': 128
        },
        'Large Batch': {
            'EMBED_DIM': 512, 'NUM_LAYERS': 12, 'NUM_HEADS': 8,
            'LEARNING_RATE': 2e-3, 'BATCH_SIZE': 256
        },
        'High LR': {
            'EMBED_DIM': 512, 'NUM_LAYERS': 12, 'NUM_HEADS': 8,
            'LEARNING_RATE': 3e-3, 'BATCH_SIZE': 128
        }
    }

    results = []

    for config_name, params in configs.items():
        print(f"\n{'='*60}")
        print(f"Testing Configuration: {config_name}")
        print(f"{'='*60}")

        config = Config()
        for key, value in params.items():
            if hasattr(config, key):
                setattr(config, key, value)

        # Train for fewer epochs for comparison
        config.EPOCHS = 50

        history, best_acc = train_model('CIFAR10', config)

        results.append({
            'Configuration': config_name,
            'Embed Dim': params['EMBED_DIM'],
            'Layers': params['NUM_LAYERS'],
            'Heads': params['NUM_HEADS'],
            'Learning Rate': params['LEARNING_RATE'],
            'Batch Size': params['BATCH_SIZE'],
            'Best Accuracy': f"{best_acc:.2f}%",
            'Final Train Acc': f"{history['train_acc'][-1]:.2f}%",
            'Final Test Acc': f"{history['test_acc'][-1]:.2f}%"
        })

    # Display Comparison Table
    print("\n" + "="*100)
    print("HYPERPARAMETER COMPARISON TABLE")
    print("="*100)
    print(f"{'Configuration':<15} {'Embed':<8} {'Layers':<8} {'Heads':<8} {'LR':<12} {'Batch':<8} {'Best Acc':<12} {'Final Train':<12} {'Final Test':<12}")
    print("-"*100)

    for result in results:
        print(f"{result['Configuration']:<15} {result['Embed Dim']:<8} {result['Layers']:<8} {result['Heads']:<8} {result['Learning Rate']:<12} {result['Batch Size']:<8} {result['Best Accuracy']:<12} {result['Final Train Acc']:<12} {result['Final Test Acc']:<12}")

    print("="*100)

    return results

# Execute Training
if __name__ == '__main__':
    # Train on CIFAR-10
    print("\nTraining on CIFAR-10...")
    history_cifar10, best_acc_cifar10 = train_model('CIFAR10')
    plot_history(history_cifar10, 'CIFAR-10')

    # Train on CIFAR-100
    print("\nTraining on CIFAR-100...")
    history_cifar100, best_acc_cifar100 = train_model('CIFAR100')
    plot_history(history_cifar100, 'CIFAR-100')

    # Compare Hyperparameters
    print("\nComparing Hyperparameters...")
    comparison_results = compare_hyperparameters()